## <span style="color:orange">!!!!!!!!!!!!!!! BEFORE RUNNING THIS CODE YOU MUST HAVE YOUR .csv FILES UNDER A DIRECTORY CALLED "dataset/init_data" AND THIS DIRECTORY MUST BE IN THE SAME LEVEL WITH THIS NOTEBOOK</span>

## Required Structure
CIC_IoT_Dataset_2023_Preprocessed.ipynb
dataset/
└── init_data/
    └── *.csv

## Result
The result after running the entire code will be added to <span style="color: yellow;">"dataset/cleaned_data"</span> --> This is the cleaned data that will be used for training the model.

# Data Pre-Processing

## Imports

In [ ]:
import pandas as pd
import os
from pathlib import Path
import shutil

In [ ]:
BASE_DIR = Path.cwd() / "dataset"
DATASET_DIRECTORY = BASE_DIR / "init_data"
CLEANED_DIRECTORY = BASE_DIR / "encoded_data"
PARQUET_DIRECTORY = BASE_DIR / "parquet_data"
SCALED_DIRECTORY = BASE_DIR / "scaled_data"
FINAL_DIRECTORY = BASE_DIR / "cleaned_data"

os.makedirs(CLEANED_DIRECTORY, exist_ok=True)
os.makedirs(PARQUET_DIRECTORY, exist_ok= True)
os.makedirs(SCALED_DIRECTORY, exist_ok= True)
os.makedirs(FINAL_DIRECTORY, exist_ok= True)

In [ ]:
# This function will be used to delete the uneccessary data (outdated folder/data)
def Delete_Directory(Directory):
    if Directory.exists() and Directory.is_dir():
        shutil.rmtree(Directory)

In [ ]:
dataframes = list(DATASET_DIRECTORY.glob("*.csv"))

In [ ]:
dataframes

[WindowsPath('C:/Users/marou/CIC IoT Dataset 2023/dataset/init_data/part-00000-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv'),
 WindowsPath('C:/Users/marou/CIC IoT Dataset 2023/dataset/init_data/part-00001-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv'),
 WindowsPath('C:/Users/marou/CIC IoT Dataset 2023/dataset/init_data/part-00002-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv'),
 WindowsPath('C:/Users/marou/CIC IoT Dataset 2023/dataset/init_data/part-00003-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv'),
 WindowsPath('C:/Users/marou/CIC IoT Dataset 2023/dataset/init_data/part-00004-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv'),
 WindowsPath('C:/Users/marou/CIC IoT Dataset 2023/dataset/init_data/part-00005-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv'),
 WindowsPath('C:/Users/marou/CIC IoT Dataset 2023/dataset/init_data/part-00006-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv'),
 WindowsPath('C:/Users/marou/CIC IoT Dataset 2023/dataset/init_data/part-00007-363d1ba3-8ab5-4f96-bc25-4

## General check on one of the dataframes so i get to know how to handle their cleaning process

In [ ]:
path = os.path.join(DATASET_DIRECTORY, dataframes[0])
test_df = pd.read_csv(path)
test_df

,flow_duration,Header_Length,Protocol Type,Duration,Rate,Srate,Drate,fin_flag_number,syn_flag_number,rst_flag_number,...,Std,Tot size,IAT,Number,Magnitue,Radius,Covariance,Variance,Weight,label
0,0.000000,54.00,6.00,64.00,0.329807,0.329807,0.0,1,0,1,...,0.000000,54.00,8.334383e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,9
1,0.000000,57.04,6.33,64.00,4.290556,4.290556,0.0,0,0,0,...,2.822973,57.04,8.292607e+07,9.5,10.464666,4.010353,160.987842,0.05,141.55,20
2,0.000000,0.00,1.00,64.00,33.396799,33.396799,0.0,0,0,0,...,0.000000,42.00,8.312799e+07,9.5,9.165151,0.000000,0.000000,0.00,141.55,6
3,0.328175,76175.00,17.00,64.00,4642.133010,4642.133010,0.0,0,0,0,...,0.000000,50.00,8.301570e+07,9.5,10.000000,0.000000,0.000000,0.00,141.55,21
4,0.117320,101.73,6.11,65.91,6.202211,6.202211,0.0,0,1,0,...,23.113111,57.88,8.297300e+07,9.5,11.346876,32.716243,3016.808286,0.19,141.55,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
238682,0.000000,54.00,6.00,64.00,3.049186,3.049186,0.0,1,0,1,...,0.000000,54.00,8.334449e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,9
238683,0.000000,54.00,6.00,64.00,183.433732,183.433732,0.0,0,0,0,...,0.000000,54.00,8.331392e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,8
238684,0.000785,56.29,6.11,64.00,306.952216,306.952216,0.0,0,1,0,...,0.140764,54.21,8.308883e+07,9.5,10.395538,0.200659,0.671167,0.03,141.55,10
238685,0.000901,72.09,6.11,64.64,158.475986,158.475986,0.0,0,0,0,...,2.450404,55.48,8.333177e+07,9.5,10.456522,3.475801,55.994224,0.17,141.55,8


In [ ]:
test_df.dtypes

flow_duration      float64
Header_Length      float64
Protocol Type      float64
Duration           float64
Rate               float64
Srate              float64
Drate              float64
fin_flag_number      int64
syn_flag_number      int64
rst_flag_number      int64
psh_flag_number      int64
ack_flag_number      int64
ece_flag_number      int64
cwr_flag_number      int64
ack_count          float64
syn_count          float64
fin_count          float64
urg_count          float64
rst_count          float64
HTTP                 int64
HTTPS                int64
DNS                  int64
Telnet               int64
SMTP                 int64
SSH                  int64
IRC                  int64
TCP                  int64
UDP                  int64
DHCP                 int64
ARP                  int64
ICMP                 int64
IPv                  int64
LLC                  int64
Tot sum            float64
Min                float64
Max                float64
AVG                float64
S

In [ ]:
test_df["ack_count"]

0         1.00
1         0.00
2         0.00
3         0.00
4         0.00
          ... 
238682    1.00
238683    0.00
238684    0.00
238685    0.02
238686    0.00
Name: ack_count, Length: 238687, dtype: float64

In [ ]:
test_df["ece_flag_number"].unique()

array([0], dtype=int64)

In [ ]:
test_df.isnull().any()

flow_duration      False
Header_Length      False
Protocol Type      False
Duration           False
Rate               False
Srate              False
Drate              False
fin_flag_number    False
syn_flag_number    False
rst_flag_number    False
psh_flag_number    False
ack_flag_number    False
ece_flag_number    False
cwr_flag_number    False
ack_count          False
syn_count          False
fin_count          False
urg_count          False
rst_count          False
HTTP               False
HTTPS              False
DNS                False
Telnet             False
SMTP               False
SSH                False
IRC                False
TCP                False
UDP                False
DHCP               False
ARP                False
ICMP               False
IPv                False
LLC                False
Tot sum            False
Min                False
Max                False
AVG                False
Std                False
Tot size           False
IAT                False


In [ ]:
test_df['label'].unique()

array([ 9, 20,  6, 21, 19, 23, 12, 25, 10,  8, 13, 14,  1, 22,  4, 24, 18,
        7, 29, 16, 15, 27, 33,  5, 26,  3, 32, 11,  0,  2, 17, 30, 28, 31],
      dtype=int64)

In [ ]:
test_df['HTTP'].unique()

array([0, 1], dtype=int64)

# Actual Pre-Processing
##### Step-1: drop null valies (missing values), transform datatypes and encode categorical variables

In [ ]:
import numpy as np

for file_path in dataframes:
    # Getting the file path and reading it (one by one)
    dataframe = pd.read_csv(path)

    # Dropping null values if they exist (dropping cuz if null values exist, they wont affect the data since it has over 30 million rows combined)
    dataframe.dropna(inplace= True)

    # Now im handling the data types (the flags must be binary, the counts must be integers {easier to deal with in the background than float})
    #First flags (Must be binary)
    flag_columns = ["fin_flag_number", "syn_flag_number", "rst_flag_number", "psh_flag_number", "ack_flag_number", "ece_flag_number", "cwr_flag_number"]
    for column in flag_columns:
        # Transforming them into True and False then -> 1 , 0 respectfully. (if greater than 0 then true, if not then false); because there might be higher numbers than 1
        dataframe[column] = (dataframe[column] > 0).astype("int8")

    # Second, Protocols datatypes (must be binary)
    protocol_columns = ["HTTP", "HTTPS", "DNS", "Telnet", "SMTP", "SSH", "IRC", "TCP", "UDP", "DHCP", "ARP", "ICMP", "IPv", "LLC"]
    for column in protocol_columns:
        dataframe[column] = dataframe[column].astype("int8")

    # Now encoding process (labels must be handled with label encoding {we dont wanna enlargen our features} )
    dataframe["label"] = dataframe["label"].astype("category").cat.codes

    # Save the processed dataframe to combine them all later
    out_path = CLEANED_DIRECTORY / file_path.name
    dataframe.to_csv(out_path, index= False)


##### Step-2: Convert all cleaned files from csv format to parquet before scaling & outliers detection (for faster and safer computing and modeling later on as well)

###### We only RUN THE BELOW CHUNK ONCE (NO NEED TO DO IT TWICE, since the directory is stored on our disks)

In [ ]:
for csv_path in CLEANED_DIRECTORY.glob("*.csv"):
    #going through everyfile (cleaned csv) by getting their path first
    dataframe = pd.read_csv(csv_path)
    #convert the csv file into parquet and moving them to the correct directory (PARQUET_DIRECTORY)
    parquet_path = PARQUET_DIRECTORY / (csv_path.stem + ".parquet")
    dataframe.to_parquet(parquet_path, index= False)

Delete_Directory(CLEANED_DIRECTORY)

In [ ]:
test_file = next(PARQUET_DIRECTORY.glob("*.parquet")) # only first file (just for scanning purposess)
test_df_parquet = pd.read_parquet(test_file)

test_df_parquet

,flow_duration,Header_Length,Protocol Type,Duration,Rate,Srate,Drate,fin_flag_number,syn_flag_number,rst_flag_number,...,Std,Tot size,IAT,Number,Magnitue,Radius,Covariance,Variance,Weight,label
0,0.000000,54.00,6.00,64.00,0.329807,0.329807,0.0,1,0,1,...,0.000000,54.00,8.334383e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,9
1,0.000000,57.04,6.33,64.00,4.290556,4.290556,0.0,0,0,0,...,2.822973,57.04,8.292607e+07,9.5,10.464666,4.010353,160.987842,0.05,141.55,20
2,0.000000,0.00,1.00,64.00,33.396799,33.396799,0.0,0,0,0,...,0.000000,42.00,8.312799e+07,9.5,9.165151,0.000000,0.000000,0.00,141.55,6
3,0.328175,76175.00,17.00,64.00,4642.133010,4642.133010,0.0,0,0,0,...,0.000000,50.00,8.301570e+07,9.5,10.000000,0.000000,0.000000,0.00,141.55,21
4,0.117320,101.73,6.11,65.91,6.202211,6.202211,0.0,0,1,0,...,23.113111,57.88,8.297300e+07,9.5,11.346876,32.716243,3016.808286,0.19,141.55,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
238682,0.000000,54.00,6.00,64.00,3.049186,3.049186,0.0,1,0,1,...,0.000000,54.00,8.334449e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,9
238683,0.000000,54.00,6.00,64.00,183.433732,183.433732,0.0,0,0,0,...,0.000000,54.00,8.331392e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,8
238684,0.000785,56.29,6.11,64.00,306.952216,306.952216,0.0,0,1,0,...,0.140764,54.21,8.308883e+07,9.5,10.395538,0.200659,0.671167,0.03,141.55,10
238685,0.000901,72.09,6.11,64.64,158.475986,158.475986,0.0,0,0,0,...,2.450404,55.48,8.333177e+07,9.5,10.456522,3.475801,55.994224,0.17,141.55,8


##### Step-3: Scaling

In [ ]:
#Before scaling, i must identitfy first the numerical columns (no target column like the (label) column ) and chehck the features to know what to scale
test_df_parquet.dtypes.unique()

array([dtype('float64'), dtype('int64')], dtype=object)

###### FIRST, exclude the label column (feature)

In [ ]:
numerical_columns = test_df_parquet.select_dtypes(include=["float64", "int64"]).columns.tolist()
numerical_columns.remove("label")

###### Now, the binary features (Protocols, flags)

In [ ]:
flag_columns = [ "fin_flag_number", "syn_flag_number", "rst_flag_number", "psh_flag_number", "ack_flag_number", "ece_flag_number", "cwr_flag_number"]
protocol_columns = ["HTTP", "HTTPS", "DNS", "Telnet", "SMTP", "SSH", "IRC", "TCP", "UDP", "DHCP", "ARP", "ICMP", "IPv", "LLC"]
binary_columns = flag_columns + protocol_columns

#now i remove the binary columns from the numerical_columns that im gonna scale
for col in binary_columns:
    if col in numerical_columns:
        numerical_columns.remove(col)

In [ ]:
numerical_columns

['flow_duration',
 'Header_Length',
 'Protocol Type',
 'Duration',
 'Rate',
 'Srate',
 'Drate',
 'ack_count',
 'syn_count',
 'fin_count',
 'urg_count',
 'rst_count',
 'Tot sum',
 'Min',
 'Max',
 'AVG',
 'Std',
 'Tot size',
 'IAT',
 'Number',
 'Magnitue',
 'Radius',
 'Covariance',
 'Variance',
 'Weight']

In [ ]:
test_df_parquet["Protocol Type"].unique()

array([ 6.  ,  6.33,  1.  , ...,  5.61,  0.  , 13.11])

In [ ]:
#Scaling this feature (Protocol Type) should be okay since it is oringally float and it probably shows & has magnitude

#### The Actual Scaling now (I'll perform StandardScaler)

In [ ]:
from sklearn.preprocessing import StandardScaler

std_scaler = StandardScaler()
for file_path in PARQUET_DIRECTORY.glob("*.parquet"):
    dataframe = pd.read_parquet(file_path)
    # I dont wanna load everything into memory so im using partial_fit instead of fit
    std_scaler.partial_fit(dataframe[numerical_columns])

In [ ]:
#Now last and final stage is to apply the scaling and save them
for file_path in PARQUET_DIRECTORY.glob("*.parquet"):
    #reading the files
    dataframe = pd.read_parquet(file_path)

    #applying the scaling
    dataframe[numerical_columns] = std_scaler.transform(dataframe[numerical_columns])

    #Saving the scaled files in the prevously made directory (SCALED_DIRECOTRY)
    destination_path = SCALED_DIRECTORY / file_path.name
    dataframe.to_parquet(destination_path, index=False)

Delete_Directory(PARQUET_DIRECTORY)

##### Checking the scaling results

In [ ]:
test_file = next(SCALED_DIRECTORY.glob("*.parquet")) # only first file (just for scanning purposess)
test_scaled_dataframe = pd.read_parquet(test_file)

test_scaled_dataframe.head()

,flow_duration,Header_Length,Protocol Type,Duration,Rate,Srate,Drate,fin_flag_number,syn_flag_number,rst_flag_number,...,Std,Tot size,IAT,Number,Magnitue,Radius,Covariance,Variance,Weight,label
0,-0.018025,-0.167425,-0.342900,-0.167212,-0.093112,-0.093112,-0.003051,1,0,1,...,-0.205885,-0.291610,0.011200,0.003291,-0.317348,-0.205716,-0.071405,-0.414431,0.003297,9
1,-0.018025,-0.167418,-0.305831,-0.167212,-0.093072,-0.093072,-0.003051,0,0,0,...,-0.188656,-0.279191,-0.013219,0.003291,-0.308990,-0.188409,-0.071046,-0.200851,0.003297,20
2,-0.018025,-0.167542,-0.904559,-0.167212,-0.092784,-0.092784,-0.003051,0,0,0,...,-0.205885,-0.340632,-0.001416,0.003291,-0.459093,-0.205716,-0.071405,-0.414431,0.003297,6
3,-0.017024,-0.001817,0.892748,-0.167212,-0.047055,-0.047055,-0.003051,0,0,0,...,-0.205885,-0.307950,-0.007980,0.003291,-0.362662,-0.205716,-0.071405,-0.414431,0.003297,21
4,-0.017667,-0.167321,-0.330544,-0.031029,-0.093053,-0.093053,-0.003051,0,1,0,...,-0.064825,-0.275759,-0.010476,0.003291,-0.207089,-0.064525,-0.064669,0.397172,0.003297,19


In [ ]:
test_scaled_dataframe[flag_columns]

,fin_flag_number,syn_flag_number,rst_flag_number,psh_flag_number,ack_flag_number,ece_flag_number,cwr_flag_number
0,1,0,1,0,0,0,0
1,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0
4,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...
238682,1,0,1,0,0,0,0
238683,0,0,0,1,1,0,0
238684,0,1,0,0,0,0,0
238685,0,0,0,1,1,0,0


In [ ]:
test_scaled_dataframe[protocol_columns]

,HTTP,HTTPS,DNS,Telnet,SMTP,SSH,IRC,TCP,UDP,DHCP,ARP,ICMP,IPv,LLC
0,0,0,0,0,0,0,0,1,0,0,0,0,1,1
1,1,0,0,0,0,0,0,1,0,0,0,0,1,1
2,0,0,0,0,0,0,0,0,0,0,0,1,1,1
3,0,0,0,0,0,0,0,0,1,0,0,0,1,1
4,0,0,0,0,0,0,0,1,0,0,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
238682,0,0,0,0,0,0,0,1,0,0,0,0,1,1
238683,0,0,0,0,0,0,0,1,0,0,0,0,1,1
238684,0,0,0,0,0,0,0,1,0,0,0,0,1,1
238685,0,0,0,0,0,0,0,1,0,0,0,0,1,1


##### Step-4: Handling Outliers

##### We cannot delete any data points because outliers in this specefic dataset are attacks, so we should instead clip them using percentiles
###### Which basically mean that we are shrnking the extreme datapoints and bringing them closer to the lower and upper bounderies (and these bounderies will be calculated for all 169 files and then we get their average) which are called global bounderies
<span style="color:red">We should only use the numerical columns and never touch the label and the binary features; because they are continous values unlike the binary, which is only a classification, same for the label (target column)</span>

In [ ]:
lower_bound = None
upper_bound = None
files_count = 0

for file_path in SCALED_DIRECTORY.glob("*.parquet"):
    #loading the data (one by one)
    dataframe = pd.read_parquet(file_path)

    lower_quantile = dataframe[numerical_columns].quantile(0.01)
    upper_quantile = dataframe[numerical_columns].quantile(0.99)

    #if its the first iteration, then just set the lower_bound to the lower_quantile and same for upper, otherwise, add them up
    if(lower_bound is None):
        lower_bound = lower_quantile
        upper_bound = upper_quantile
    else:
        lower_bound += lower_quantile
        upper_bound += upper_quantile

    files_count += 1

lower_bound = lower_bound / files_count
upper_bound = upper_bound

In [ ]:
lower_bound.head()

flow_duration   -0.018025
Header_Length   -0.167542
Protocol Type   -0.905682
Duration        -0.271310
Rate            -0.093115
Name: 0.01, dtype: float64

In [ ]:
upper_bound.head()

flow_duration     29.758394
Header_Length    801.403306
Protocol Type    720.396334
Duration         882.866686
Rate             204.722514
Name: 0.99, dtype: float64

##### Clipping application

In [ ]:
# Clipping loop
for file_path in SCALED_DIRECTORY.glob("*.parquet"):
    dataframe = pd.read_parquet(file_path)

    #Clipping
    dataframe[numerical_columns] = dataframe[numerical_columns].clip(lower= lower_bound, upper= upper_bound, axis=1)

    #saving the final pre-processed files in the FINAL_DIRECTORY
    destination_path = FINAL_DIRECTORY / file_path.name
    dataframe.to_parquet(destination_path, index= False)

Delete_Directory(SCALED_DIRECTORY)

In [ ]:
path = next(FINAL_DIRECTORY.glob("*.parquet"))
processed_df = pd.read_parquet(path)

processed_df.head()

,flow_duration,Header_Length,Protocol Type,Duration,Rate,Srate,Drate,fin_flag_number,syn_flag_number,rst_flag_number,...,Std,Tot size,IAT,Number,Magnitue,Radius,Covariance,Variance,Weight,label
0,-0.018025,-0.167425,-0.342900,-0.167212,-0.093112,-0.093112,-0.51562,1,0,1,...,-0.205885,-0.291610,0.011200,0.003291,-0.317348,-0.205716,-0.071405,-0.414431,0.003297,9
1,-0.018025,-0.167418,-0.305831,-0.167212,-0.093072,-0.093072,-0.51562,0,0,0,...,-0.188656,-0.279191,-0.013219,0.003291,-0.308990,-0.188409,-0.071046,-0.200851,0.003297,20
2,-0.018025,-0.167542,-0.904559,-0.167212,-0.092784,-0.092784,-0.51562,0,0,0,...,-0.205885,-0.340632,-0.001416,0.003291,-0.459093,-0.205716,-0.071405,-0.414431,0.003297,6
3,-0.017024,-0.001817,0.892748,-0.167212,-0.047055,-0.047055,-0.51562,0,0,0,...,-0.205885,-0.307950,-0.007980,0.003291,-0.362662,-0.205716,-0.071405,-0.414431,0.003297,21
4,-0.017667,-0.167321,-0.330544,-0.031029,-0.093053,-0.093053,-0.51562,0,1,0,...,-0.064825,-0.275759,-0.010476,0.003291,-0.207089,-0.064525,-0.064669,0.397172,0.003297,19
